# MALTO — v4: ModernBERT-large + FGM Adversarial Training

**Goal:** Beat 0.96423 (1st place) → target 0.97+

**Key changes over v1 (0.95341):**
1. **ModernBERT-large** (395M params, flash attention) — bigger model, richer representations
2. **FGM adversarial training** (ε=0.3) — perturbs embeddings after each backward to force robust minority-class features
3. **DRW cap reverted to 10×** — 20× proved unstable (v2 fold variance 0.0128 vs 0.0044)
4. **SWA top-3 checkpoints** per fold — smoother decision boundaries
5. **Per-class Nelder-Mead ensemble** — prevents SVC (F1=0.51 on DeepSeek) from hurting blend
6. **Wider threshold sweep [0.60, 1.50]** — more room to find minority-class multipliers

**Runtime:** ~240 min on T4×2
**Run in parallel with:** solution_v5_deberta_large.ipynb


In [ ]:
import os, warnings, gc, time, json, re
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Reduce memory fragmentation (PyTorch recommended for OOM)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    N_GPUS  = torch.cuda.device_count()
    USE_AMP = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32       = False
    print(f'GPU: {torch.cuda.get_device_name(0)} x {N_GPUS}')
    print(f'AMP: {USE_AMP} | TF32: disabled (prevents NaN)')
else:
    DEVICE = torch.device('cpu')
    N_GPUS  = 1
    USE_AMP = False

KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input/datasets/aliivaezii/test-and-train',
    '/kaggle/input', '.', '..'
]
DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p
        break
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root
            break
assert DATA_DIR is not None, f'train.csv not found!'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
train_df['TEXT'] = train_df['TEXT'].fillna('empty')
test_df['TEXT']  = test_df['TEXT'].fillna('empty')

y_all       = train_df['LABEL'].values
texts_train = train_df['TEXT'].values
texts_test  = test_df['TEXT'].values

print(f'Data dir: {DATA_DIR}')
print(f'Device: {DEVICE} | GPUs: {N_GPUS} | AMP: {USE_AMP}')
print(f'Train: {len(train_df)}, Test: {len(test_df)}')


In [ ]:
# ============================================================
# CONFIG — ModernBERT-large
# ============================================================
MODEL_NAME  = 'answerdotai/ModernBERT-large'
MAX_LEN     = 512
BATCH_SIZE  = 8 * N_GPUS     # 8 per GPU (fits 15 GB VRAM)
EPOCHS      = 10
LR          = 2e-5            # lower than base; large models need smaller LR
PATIENCE    = 3
N_FOLDS     = 5
NUM_WORKERS = 4

# v4 improvements
DRW_CAP     = 10              # 10x safe (20x increased fold variance in v2)
TOP_K_CKPTS = 3               # SWA: average top-K checkpoints per fold
FGM_EPSILON = 0.3             # adversarial perturbation magnitude
EMB_NAME    = 'tok_embeddings'  # ModernBERT embedding layer name for FGM

print(f'Model:      {MODEL_NAME}')
print(f'Batch:      {BATCH_SIZE} ({BATCH_SIZE//N_GPUS} per GPU × {N_GPUS} GPUs)')
print(f'LR:         {LR}  |  Folds: {N_FOLDS}  |  Epochs: {EPOCHS}  |  Patience: {PATIENCE}')
print(f'DRW cap:    {DRW_CAP}x  |  SWA top-{TOP_K_CKPTS}  |  FGM eps: {FGM_EPSILON}')


In [ ]:
# ============================================================
# LOSS: LDAM + Gradual DRW + Label Smoothing
# ============================================================
class LDAMLoss(nn.Module):
    def __init__(self, class_counts, max_margin=0.5,
                 drw_epoch=1, drw_ramp_epochs=3, label_smoothing=0.1):
        super().__init__()
        safe = np.maximum(class_counts, 1).astype(np.float64)
        margins = np.clip(max_margin / np.power(safe, 0.25), 0, 1.0)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))
        self.drw_epoch       = drw_epoch
        self.drw_ramp_epochs = drw_ramp_epochs
        self.current_epoch   = 0
        self.label_smoothing = label_smoothing

        inv = 1.0 / safe
        w   = inv / inv.sum() * len(class_counts)
        w   = np.clip(w, w.min(), w.min() * DRW_CAP)
        w   = w / w.sum() * len(class_counts)
        w   = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        self.register_buffer('drw_weights', torch.tensor(w, dtype=torch.float32))

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def _blended_weight(self, device):
        ep = self.current_epoch
        if ep < self.drw_epoch:
            return None
        t    = min((ep - self.drw_epoch) / max(self.drw_ramp_epochs, 1), 1.0)
        ones = torch.ones_like(self.drw_weights)
        return (ones + t * (self.drw_weights - ones)).to(device)

    def forward(self, logits, targets):
        logits = logits.float()
        if torch.isnan(logits).any():
            logits = torch.nan_to_num(logits, nan=0.0)
        margins  = self.margins.to(logits.device)[targets]
        adjusted = logits.clone()
        adjusted[torch.arange(len(targets), device=logits.device), targets] -= margins
        weight = self._blended_weight(logits.device)
        loss   = F.cross_entropy(adjusted, targets, weight=weight,
                                 label_smoothing=self.label_smoothing)
        if torch.isnan(loss):
            return F.cross_entropy(logits, targets)
        return loss

# ============================================================
# FGM — Fast Gradient Method adversarial training
# ============================================================
class FGM:
    """Perturbs embedding parameters in the gradient direction by epsilon.
    Calling attack() then running a second forward+backward exposes the model
    to adversarial inputs, helping it learn more robust minority-class features.
    """
    def __init__(self, model, epsilon=0.3):
        self.model   = model
        self.epsilon = epsilon
        self.backup  = {}

    def attack(self, emb_name='embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = param.grad.norm()
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name='embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

print(f'LDAMLoss ready | DRW cap={DRW_CAP}x | FGM ready | epsilon={FGM_EPSILON}')


In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================
print(f'Tokenizing with {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(list(texts_train), max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')
test_enc  = tokenizer(list(texts_test),  max_length=MAX_LEN, padding='max_length',
                      truncation=True, return_tensors='pt')

print(f'Train: {train_enc["input_ids"].shape}, Test: {test_enc["input_ids"].shape}')
t_start = time.time()


In [ ]:
class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, indices=None):
        self.encodings = encodings
        self.labels    = labels
        self.indices   = indices

    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        real_idx = self.indices[idx] if self.indices is not None else idx
        item = {k: v[real_idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[real_idx], dtype=torch.long)
        return item


def get_optimizer(model, lr, llrd=0.9):
    """AdamW with layer-wise LR decay. Handles ModernBERT and DeBERTa layer naming."""
    no_decay = ['bias', 'LayerNorm', 'layernorm', 'layer_norm']
    params   = []

    head = [(n, p) for n, p in model.named_parameters()
            if 'classifier' in n or 'head' in n]
    if head:
        params.append({'params': [p for n, p in head if not any(nd in n for nd in no_decay)],
                       'lr': lr, 'weight_decay': 0.01})
        params.append({'params': [p for n, p in head if     any(nd in n for nd in no_decay)],
                       'lr': lr, 'weight_decay': 0.0})

    all_names  = [n for n, _ in model.named_parameters()]
    layer_nums = set()
    for n in all_names:
        m = re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m:
            layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 24

    for i in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd ** (num_layers - 1 - i))
        patterns = [f'encoder.layer.{i}.', f'layers.{i}.']
        layer_params = [(n, p) for n, p in model.named_parameters()
                        if any(pat in n for pat in patterns)]
        if layer_params:
            wd_p   = [p for n, p in layer_params if not any(nd in n for nd in no_decay)]
            nowd_p = [p for n, p in layer_params if     any(nd in n for nd in no_decay)]
            if wd_p:   params.append({'params': wd_p,   'lr': layer_lr, 'weight_decay': 0.01})
            if nowd_p: params.append({'params': nowd_p, 'lr': layer_lr, 'weight_decay': 0.0})

    emb_lr     = lr * (llrd ** num_layers)
    emb_params = [(n, p) for n, p in model.named_parameters() if 'embed' in n.lower()]
    if emb_params:
        wd_p   = [p for n, p in emb_params if not any(nd in n for nd in no_decay)]
        nowd_p = [p for n, p in emb_params if     any(nd in n for nd in no_decay)]
        if wd_p:   params.append({'params': wd_p,   'lr': emb_lr, 'weight_decay': 0.01})
        if nowd_p: params.append({'params': nowd_p, 'lr': emb_lr, 'weight_decay': 0.0})

    assigned = set(id(p) for grp in params for p in grp['params'])
    rem = [p for n, p in model.named_parameters() if id(p) not in assigned and p.requires_grad]
    if rem:
        params.append({'params': rem, 'lr': lr * 0.5, 'weight_decay': 0.01})

    return torch.optim.AdamW(params)

print('Dataset & optimizer ready.')


In [ ]:
# ============================================================
# K-FOLD TRAINING  (LDAM + FGM adversarial + SWA top-K)
# ============================================================
def train_fold(fold_idx, train_idx, val_idx):
    gc.collect(); torch.cuda.empty_cache()
    print(f'\n{"="*50}')
    print(f'FOLD {fold_idx+1}/{N_FOLDS} | Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*50}')

    train_loader = DataLoader(TextDataset(train_enc, y_all, train_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=True)
    val_loader  = DataLoader(TextDataset(train_enc, y_all, val_idx),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(TextDataset(test_enc),
        batch_size=BATCH_SIZE*2, num_workers=NUM_WORKERS, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
    # Gradient checkpointing: recompute activations during backward (~50-70% less activation memory)
    # Critical for FGM which runs 2 backward passes per step on a 395M param model
    base = model.model if hasattr(model, 'model') else model
    base.gradient_checkpointing_enable()
    if N_GPUS > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts, max_margin=0.5, drw_epoch=1, drw_ramp_epochs=3)
    criterion.to(DEVICE)
    print(f'  LDAM margins:  {criterion.margins.cpu().numpy().round(3)}')
    print(f'  DRW weights:   {criterion.drw_weights.cpu().numpy().round(2)}')

    actual_model = model.module if hasattr(model, 'module') else model
    optimizer = get_optimizer(actual_model, lr=LR, llrd=0.9)
    fgm       = FGM(actual_model, epsilon=FGM_EPSILON)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps)
    scaler = GradScaler(enabled=USE_AMP)

    top_checkpoints  = []
    best_f1          = 0
    patience_counter = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)
        if epoch < criterion.drw_epoch:
            drw_pct = '  0%'
        else:
            t = min((epoch - criterion.drw_epoch) / criterion.drw_ramp_epochs, 1.0)
            drw_pct = f'{int(t*100):3d}%'

        model.train()
        total_loss, valid_steps = 0.0, 0

        for batch in tqdm(train_loader, desc=f'E{epoch+1}', leave=False):
            optimizer.zero_grad()
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')

            # ── Normal forward + backward ───────────────────
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
                loss    = criterion(outputs.logits, labels)

            if torch.isnan(loss) or torch.isinf(loss):
                scheduler.step()
                continue

            scaler.scale(loss).backward()

            # ── FGM: perturb embeddings, adversarial pass ───
            fgm.attack(EMB_NAME)
            with autocast(enabled=USE_AMP):
                adv_outputs = model(**inputs)
                adv_loss    = criterion(adv_outputs.logits, labels)
            if not (torch.isnan(adv_loss) or torch.isinf(adv_loss)):
                scaler.scale(adv_loss).backward()
            fgm.restore(EMB_NAME)

            # ── Gradient clip + step ─────────────────────────
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss  += loss.item()
            valid_steps += 1
            scheduler.step()

        avg_loss = total_loss / max(valid_steps, 1)

        model.eval()
        preds, labels_list = [], []
        with torch.no_grad():
            for batch in val_loader:
                inp = {k: v.to(DEVICE) for k, v in batch.items()}
                lbl = inp.pop('labels')
                with autocast(enabled=USE_AMP):
                    out = model(**inp)
                preds.extend(out.logits.argmax(-1).cpu().numpy())
                labels_list.extend(lbl.cpu().numpy())

        val_f1  = f1_score(labels_list, preds, average='macro')
        elapsed = time.time() - t0

        cur_model = model.module if hasattr(model, 'module') else model
        state = {k: v.cpu().clone() for k, v in cur_model.state_dict().items()}
        if len(top_checkpoints) < TOP_K_CKPTS or val_f1 > top_checkpoints[-1][0]:
            if len(top_checkpoints) >= TOP_K_CKPTS:
                top_checkpoints.pop()
            top_checkpoints.append((val_f1, state))
            top_checkpoints.sort(key=lambda x: x[0], reverse=True)

        if val_f1 > best_f1:
            best_f1 = val_f1; patience_counter = 0; marker = ' ** BEST **'
        else:
            patience_counter += 1; marker = f' (p={patience_counter}/{PATIENCE})'

        print(f'  E{epoch+1} | loss={avg_loss:.4f} | val_f1={val_f1:.4f} | DRW={drw_pct} | {elapsed:.0f}s{marker}')
        if patience_counter >= PATIENCE:
            print(f'  Early stop at E{epoch+1}'); break

    # ── Average top-K checkpoints (SWA-like) ────────────────
    n_avg = len(top_checkpoints)
    print(f'  Averaging top-{n_avg} checkpoints: {[f"{f:.4f}" for f,_ in top_checkpoints]}')
    avg_state = {k: (sum(c[k].float() for _,c in top_checkpoints)/n_avg
                     ).to(top_checkpoints[0][1][k].dtype)
                 for k in top_checkpoints[0][1]}
    cur_model = model.module if hasattr(model, 'module') else model
    cur_model.load_state_dict(avg_state)
    model.eval()

    val_logits, test_logits_fold = [], []
    with torch.no_grad():
        for batch in val_loader:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}; inp.pop('labels', None)
            with autocast(enabled=USE_AMP): out = model(**inp)
            val_logits.extend(out.logits.float().cpu().numpy())
        for batch in test_loader:
            inp = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP): out = model(**inp)
            test_logits_fold.extend(out.logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler, top_checkpoints, avg_state
    gc.collect(); torch.cuda.empty_cache()
    return np.array(val_logits), np.array(test_logits_fold), best_f1

print(f'Training ready | LDAM+DRW({DRW_CAP}x) | LLRD=0.9 | SWA(top-{TOP_K_CKPTS}) | FGM(eps={FGM_EPSILON}) | grad_ckpt=ON')


In [ ]:
# ============================================================
# RUN 5-FOLD TRAINING
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_logits       = np.zeros((len(y_all), NUM_LABELS))
test_logits_sum  = np.zeros((len(texts_test), NUM_LABELS))
fold_scores      = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    val_logits, test_logits_fold, best_f1 = train_fold(fold_idx, train_idx, val_idx)
    oof_logits[val_idx] = val_logits
    test_logits_sum    += test_logits_fold
    fold_scores.append(best_f1)
    print(f'  Fold {fold_idx+1} best: {best_f1:.4f} | Total time: {(time.time()-t_start)/60:.1f}min')

test_logits_avg = test_logits_sum / N_FOLDS

print(f'\n{"="*50}')
print(f'{N_FOLDS}-FOLD COMPLETE')
print(f'  Scores: {[f"{s:.4f}" for s in fold_scores]}')
print(f'  Mean:   {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'  Time:   {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')


In [ ]:
# ============================================================
# TEMPERATURE SCALING  (calibrate ModernBERT logits)
# ============================================================
logits_t = torch.tensor(oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all,     dtype=torch.long)

best_temp = 1.0
best_nll  = float('inf')
for temp in np.arange(0.3, 3.0, 0.05):
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll:
        best_nll  = nll
        best_temp = temp

print(f'Optimal temperature: {best_temp:.2f}')

oof_probs  = torch.softmax(logits_t / best_temp, dim=-1).numpy()
test_probs = torch.softmax(torch.tensor(test_logits_avg, dtype=torch.float32) / best_temp, dim=-1).numpy()

oof_f1 = f1_score(y_all, oof_probs.argmax(1), average='macro')
print(f'OOF F1 (ModernBERT): {oof_f1:.4f}')
print()
print(classification_report(y_all, oof_probs.argmax(1),
      target_names=[LABEL_MAP[i] for i in range(NUM_LABELS)]))


In [ ]:
# ============================================================
# TF-IDF + CALIBRATED SVC
# ============================================================
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack as sparse_hstack

print('Building TF-IDF features...')
t_svc = time.time()

char_tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)
word_tfidf = TfidfVectorizer(analyzer='word',    ngram_range=(1, 2),
                              max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True)

char_train = char_tfidf.fit_transform(texts_train)
char_test  = char_tfidf.transform(texts_test)
word_train = word_tfidf.fit_transform(texts_train)
word_test  = word_tfidf.transform(texts_test)

X_sparse_train = sparse_hstack([char_train, word_train])
X_sparse_test  = sparse_hstack([char_test,  word_test])
print(f'  Features: {X_sparse_train.shape[1]:,}')

skf_svc = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
svc_oof_probs  = np.zeros((len(y_all), NUM_LABELS))
svc_test_probs = np.zeros((len(texts_test), NUM_LABELS))

for fold_idx, (tr, va) in enumerate(skf_svc.split(np.zeros(len(y_all)), y_all)):
    clf = CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000), cv=3, method='sigmoid')
    clf.fit(X_sparse_train[tr], y_all[tr])
    svc_oof_probs[va]  = clf.predict_proba(X_sparse_train[va])
    svc_test_probs    += clf.predict_proba(X_sparse_test) / N_FOLDS
    f1 = f1_score(y_all[va], clf.predict(X_sparse_train[va]), average='macro')
    print(f'  SVC Fold {fold_idx+1}: F1={f1:.4f}')

svc_oof_f1 = f1_score(y_all, svc_oof_probs.argmax(1), average='macro')
print(f'\nSVC OOF F1: {svc_oof_f1:.4f}  ({time.time()-t_svc:.0f}s)')
print(classification_report(y_all, svc_oof_probs.argmax(1),
      target_names=[LABEL_MAP[i] for i in range(NUM_LABELS)]))


In [ ]:
# ============================================================
# ENSEMBLE: Per-class Nelder-Mead  (ModernBERT + SVC)
# ============================================================
# No full-data model blend here (saved time for 5-fold FGM training).
# test_probs is already the 5-fold average — more stable than a single run.
from scipy.optimize import minimize

print('Ensemble weight optimization...')

# ── Strategy 1: Global weights ──────────────────────────────
def neg_macro_f1(weights, probs_list, labels):
    w = np.abs(weights); w = w / w.sum()
    blended = sum(wi * pi for wi, pi in zip(w, probs_list))
    return -f1_score(labels, blended.argmax(1), average='macro')

oof_components = [oof_probs, svc_oof_probs]
best_result = None
for init in [[0.85, 0.15], [0.80, 0.20], [0.90, 0.10],
             [0.75, 0.25], [0.70, 0.30], [0.95, 0.05]]:
    r = minimize(neg_macro_f1, init, args=(oof_components, y_all),
                 method='Nelder-Mead', options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-6})
    if best_result is None or r.fun < best_result.fun:
        best_result = r

opt_w = np.abs(best_result.x) / np.abs(best_result.x).sum()
oof_blend_global = opt_w[0]*oof_probs + opt_w[1]*svc_oof_probs
global_f1 = f1_score(y_all, oof_blend_global.argmax(1), average='macro')
print(f'  Global weights:  MB={opt_w[0]:.3f}, SVC={opt_w[1]:.3f}  → OOF F1={global_f1:.4f}')

# ── Strategy 2: Per-class SVC weights ────────────────────────
def neg_macro_f1_perclass(w_svc, mb_probs, svc_probs, labels):
    w = np.clip(w_svc, 0.0, 0.5)
    blended = np.array([(1-w[c])*mb_probs[:,c] + w[c]*svc_probs[:,c]
                         for c in range(NUM_LABELS)]).T
    return -f1_score(labels, blended.argmax(1), average='macro')

np.random.seed(SEED)
best_pc = None
for _ in range(12):
    init = np.random.uniform(0.05, 0.45, size=NUM_LABELS)
    r = minimize(neg_macro_f1_perclass, init, args=(oof_probs, svc_oof_probs, y_all),
                 method='Nelder-Mead', options={'maxiter': 10000, 'xatol': 1e-5, 'fatol': 1e-7})
    if best_pc is None or r.fun < best_pc.fun:
        best_pc = r

opt_w_pc = np.clip(best_pc.x, 0.0, 0.5)
oof_blend_pc = np.array([(1-opt_w_pc[c])*oof_probs[:,c] + opt_w_pc[c]*svc_oof_probs[:,c]
                           for c in range(NUM_LABELS)]).T
perclass_f1 = f1_score(y_all, oof_blend_pc.argmax(1), average='macro')
print(f'  Per-class SVC:   {dict(zip(LABEL_MAP.values(), opt_w_pc.round(3)))}')
print(f'  Per-class OOF F1: {perclass_f1:.4f}')

# ── Pick best ────────────────────────────────────────────────
use_perclass = perclass_f1 >= global_f1
if use_perclass:
    oof_blend     = oof_blend_pc
    oof_blend_f1  = perclass_f1
    test_blend    = np.array([(1-opt_w_pc[c])*test_probs[:,c] + opt_w_pc[c]*svc_test_probs[:,c]
                               for c in range(NUM_LABELS)]).T
    print(f'  >> Using PER-CLASS blend (+{perclass_f1-global_f1:.4f})')
else:
    oof_blend     = oof_blend_global
    oof_blend_f1  = global_f1
    test_blend    = opt_w[0]*test_probs + opt_w[1]*svc_test_probs
    print(f'  >> Using GLOBAL blend')

print(f'\n  ModernBERT OOF F1:  {oof_f1:.4f}')
print(f'  SVC OOF F1:         {svc_oof_f1:.4f}')
print(f'  Blended OOF F1:     {oof_blend_f1:.4f}')

# Test distribution
blend_preds = test_blend.argmax(1)
print(f'\nTest distribution:')
for i, nm in LABEL_MAP.items():
    print(f'  {nm:10s}: {(blend_preds==i).sum()}')


In [ ]:
# ============================================================
# THRESHOLD OPTIMIZATION  [0.60, 1.50]
# ============================================================
def threshold_sweep(oof_probs_2400, labels, n_classes=6, steps=60, passes=8):
    """Coordinate descent over per-class multipliers. Wider [0.60, 1.50] range
    lets the model shift decision boundaries for minority classes."""
    best_mult = np.ones(n_classes)
    best_f1   = f1_score(labels, oof_probs_2400.argmax(1), average='macro')
    lo, hi    = 0.60, 1.50
    for _ in range(passes):
        improved = False
        for cls in range(n_classes):
            for m in np.linspace(lo, hi, steps):
                trial      = best_mult.copy()
                trial[cls] = m
                f1 = f1_score(labels, (oof_probs_2400 * trial).argmax(1), average='macro')
                if f1 > best_f1 + 1e-6:
                    best_f1, best_mult, improved = f1, trial.copy(), True
        if not improved:
            break
    return best_mult, best_f1

thresholds, thresh_f1 = threshold_sweep(oof_blend, y_all)
print(f'OOF F1 after threshold nudge: {thresh_f1:.4f}  (was {oof_blend_f1:.4f})')
print('Multipliers:', {LABEL_MAP[i]: round(thresholds[i], 3) for i in range(NUM_LABELS)})

final_preds = (test_blend * thresholds).argmax(1)


In [ ]:
# ============================================================
# CREATE SUBMISSION
# ============================================================
id_col = None
for c in test_df.columns:
    if c.lower() in ('id', 'unnamed: 0', 'index'):
        id_col = c
        break

inv_label_map = {v: k for k, v in LABEL_MAP.items()}
pred_labels   = [LABEL_MAP[p] for p in final_preds]

if id_col:
    submission = pd.DataFrame({'id': test_df[id_col], 'LABEL': pred_labels})
else:
    submission = pd.DataFrame({'LABEL': pred_labels})

submission.to_csv('submission.csv', index=False)
print('Submission saved!')
print()
print('Prediction distribution:')
for i, name in LABEL_MAP.items():
    cnt = (final_preds == i).sum()
    print(f'  {name}: {cnt} ({cnt/len(final_preds)*100:.1f}%)')

blend_type = 'per-class' if use_perclass else 'global'
print(f'\n{"="*50}')
print(f'FINAL SUMMARY')
print(f'  Model:              {MODEL_NAME}')
print(f'  ModernBERT OOF F1:  {oof_f1:.4f}')
print(f'  SVC OOF F1:         {svc_oof_f1:.4f}')
print(f'  Blended OOF F1:     {oof_blend_f1:.4f}  ({blend_type})')
print(f'  + threshold nudge:  {thresh_f1:.4f}')
print(f'  DRW cap:            {DRW_CAP}x  |  FGM eps: {FGM_EPSILON}')
print(f'  Folds: {N_FOLDS}  |  SWA top-{TOP_K_CKPTS}  |  Temperature: {best_temp:.2f}')
print(f'  Total time:         {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')
